#### Ingesting FHV Trips Data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from dotenv import load_dotenv
import logging
import os

load_dotenv()

True

##### Creating directory for fhv log file 

In [2]:
log_file = r"/app/data/logs/"
os.makedirs(name = log_file,exist_ok= True)
file_name_fhv = os.path.join(log_file,'fhv_trips.log')
print(file_name_fhv)

/app/data/logs/fhv_trips.log


##### Adding logger

In [3]:
logger = logging.getLogger(__name__)
logger.propagate = False   # <-- add this
logger.setLevel(logging.DEBUG)
fh = logging.FileHandler(file_name_fhv,mode = 'w')
logger.addHandler(fh)
formatter = logging.Formatter('[%(asctime)s] %(levelname)s: %(message)s')
fh.setFormatter(formatter)

##### Creating spark session

In [4]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("NYC Taxi Pipeline")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.5.0,"
        "software.amazon.awssdk:bundle:2.31.54"
    )
    .config("spark.driver.memory", "4g")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.endpoint", os.environ.get("S3_ENDPOINT", "s3.amazonaws.com"))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions",8)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")
print(f"Master        : {spark.sparkContext.master}")

:: loading settings :: url = jar:file:/app/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7a9e0d68-0c85-434b-b098-f7183177c440;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 267ms :: artifacts dl 7ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
	software.amazon.awssdk#bundle;2.35.4 

Spark version : 4.2.0
App name      : NYC Taxi Pipeline
Master        : local[*]


##### Create Schema Definition
 

In [5]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)

fhv_schema = StructType([
    StructField("dispatching_base_num", StringType(), True),
    StructField("pickup_datetime", TimestampType(), True),
    StructField("dropOff_datetime", TimestampType(), True),
    StructField("PUlocationID", LongType(), True),
    StructField("DOlocationID", LongType(), True),
    StructField("SR_Flag", LongType(), True),
    StructField("Affiliated_base_number", StringType(), True)
])

##### Reading FHV file

In [6]:
fhv_data = spark.read\
                .option('header',True)\
                    .schema(fhv_schema)\
                        .parquet('/app/data/input/fhv/fhv_tripdata_2026-04.parquet')

In [7]:
fhv_data.printSchema()

root
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropOff_datetime: timestamp (nullable = true)
 |-- PUlocationID: long (nullable = true)
 |-- DOlocationID: long (nullable = true)
 |-- SR_Flag: long (nullable = true)
 |-- Affiliated_base_number: string (nullable = true)



In [8]:
fhv_data.show(10)

+--------------------+-------------------+-------------------+------------+------------+-------+----------------------+
|dispatching_base_num|    pickup_datetime|   dropOff_datetime|PUlocationID|DOlocationID|SR_Flag|Affiliated_base_number|
+--------------------+-------------------+-------------------+------------+------------+-------+----------------------+
|              B00014|2026-04-01 00:55:00|2026-04-01 01:23:00|        NULL|        NULL|   NULL|                B00013|
|              B00014|2026-04-01 00:00:00|2026-04-01 00:31:00|        NULL|        NULL|   NULL|                B00014|
|              B00053|2026-04-01 00:35:00|2026-04-01 00:45:00|        NULL|        NULL|   NULL|                B00053|
|              B00053|2026-04-01 00:29:00|2026-04-01 00:39:00|        NULL|        NULL|   NULL|                B00053|
|              B00053|2026-04-01 00:31:00|2026-04-01 00:32:00|        NULL|        NULL|   NULL|                B00053|
|              B00111|2026-04-01 00:18:0

##### Log Raw Count

In [9]:
logger.info(f'Raw count: {fhv_data.count()}')

##### Cleansing data outside of April 2026 Window

In [10]:
fhv_data = fhv_data.filter(~(date_format(col('pickup_datetime'),'yyyy-MM') >= '2026-05'))

In [11]:
fhv_data = fhv_data.filter(~(date_format(col('pickup_datetime'),'yyyy-MM') < '2026-04'))

##### Removing data if location data is missing

In [12]:
fhv_data = fhv_data.filter(~(col('PUlocationID').isNull() & col('DOlocationID').isNull()))

##### Checking if pickup and drop datetime are identical (data quality issue)

In [13]:
fhv_DQ_cnt = fhv_data.filter(col('pickup_datetime') >= col('dropOff_datetime')).count()

In [14]:
if fhv_DQ_cnt != 0:
    fhv_data = fhv_data.filter(~(col('pickup_datetime') == col('dropOff_datetime')))


##### Log count after validation

In [15]:
logger.info(f"After Validation Count: {fhv_data.count()}")

##### Derived Column trip duration in minutes

In [16]:
from pyspark.sql.functions import timestamp_diff
fhv_data = fhv_data.withColumn('trip_duration_minutes',timestamp_diff('minute',col('pickup_datetime'),col('dropOff_datetime')))

##### Clearing any trips with zero minutes trip duration

In [17]:
fhv_data = fhv_data.filter(~(col('trip_duration_minutes') == 0))

##### Adding source filename and ingestion timestamp columns

In [18]:
fhv_data = fhv_data.withColumns({'source_file': lit('fhv_tripdata_2026-04.parquet'),
'ingestion_timestamp' : current_timestamp()})

##### Writing data with paritioning in S3 Bucket

In [19]:
s3_bucket = os.environ["S3_BUCKET"]

fhv_data \
    .withColumn('pickup_date', date_format(col('pickup_datetime'), 'yyyy-MM-dd')) \
    .write \
    .option('header', True) \
    .partitionBy('pickup_date') \
    .mode('overwrite') \
    .parquet(f's3a://{s3_bucket}/fhv')

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


##### Data Count written to S3

In [20]:
logger.info(f'Complete data count written to S3: {fhv_data.count()}')